# Analysis
Loads, cleans, and analyzes the healthcare appointment dataset.

Computed results are pickled to `analysis_results.pkl` at the end, so `02_visualization.ipynb` can pick them up without recomputing anything.

In [53]:
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

In [41]:
# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/appointments.csv', parse_dates=['appointment_date'])
df.rename(columns={'specialty': 'doctor_specialty'}, inplace=True)

print("=== Dataset Overview ===")
print(f"Total Records : {len(df)}")
print(f"Date Range    : {df['appointment_date'].min().date()} → {df['appointment_date'].max().date()}")
print(f"Divisions     : {df['division'].nunique()}")
print(f"Specialties   : {df['doctor_specialty'].nunique()}\n")

print("=== Column Data Types ===")
print(df.dtypes)

print("=== First 3 Rows ===")
df.head(3)

=== Dataset Overview ===
Total Records : 500
Date Range    : 2023-01-02 → 2024-12-26
Divisions     : 8
Specialties   : 6

=== Column Data Types ===
patient_age                      int64
patient_gender                     str
division                           str
doctor_specialty                   str
appointment_status                 str
consultation_fee_bdt             int64
wait_days                        int64
appointment_date        datetime64[us]
dtype: object
=== First 3 Rows ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27


In [42]:
# ── Basic Cleaning ─────────────────────────────────────────────────────────────
print("=== Missing Values ===")
print(df.isnull().sum())

=== Missing Values ===
patient_age             0
patient_gender          0
division                0
doctor_specialty        0
appointment_status      0
consultation_fee_bdt    0
wait_days               0
appointment_date        0
dtype: int64


In [43]:
# Age group segmentation
bins = [0, 17, 35, 55, 100]
labels = ['Child (0-17)', 'Young Adult (18-35)', 'Middle Age (36-55)', 'Senior (56+)']
df['age_group'] = pd.cut(df['patient_age'], bins=bins, labels=labels)

print("=== First 3 Rows after adding Age Groups column ===")
df.head(3)

=== First 3 Rows after adding Age Groups column ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date,age_group
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19,Senior (56+)
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07,Young Adult (18-35)
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27,Senior (56+)


In [44]:
# Extract month name for trend analysis
df['month'] = df['appointment_date'].dt.month
df['year'] = df['appointment_date'].dt.year
df['month_name'] = df['appointment_date'].dt.strftime('%b')

print("=== First 3 Rows after adding Month Year columns ===")
df.head(3)

=== First 3 Rows after adding Month Year columns ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date,age_group,month,year,month_name
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19,Senior (56+),11,2023,Nov
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07,Young Adult (18-35),6,2024,Jun
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27,Senior (56+),2,2023,Feb


In [64]:
# Filter appointments using division or specialty if needed
def filter_appointments(df, division=None, specialty=None):
    filtered_df = df.copy()
    
    if division is not None:
        filtered_df = filtered_df[filtered_df['division'] == division]
        
    if specialty is not None:
        filtered_df = filtered_df[filtered_df['doctor_specialty'] == specialty]
        
    return filtered_df

### Analyses 1:
Appointment Status Breakdown 

In [66]:
filtered_df = filter_appointments(df, division="Dhaka")
status_counts = (
    filtered_df.groupby(['division', 'doctor_specialty'])['appointment_status']
    .value_counts()
    .rename('appointments_count')
    .reset_index()
)
print("=== Appointment Status ===")
display(status_counts)

=== Appointment Status ===


,division,doctor_specialty,appointment_status,appointments_count
0,Dhaka,Cardiologist,Completed,19
1,Dhaka,Cardiologist,Cancelled,5
2,Dhaka,Cardiologist,No-show,4
3,Dhaka,Dermatologist,Completed,19
4,Dhaka,Dermatologist,Cancelled,7
5,Dhaka,Dermatologist,No-show,6
6,Dhaka,General Physician,Completed,17
7,Dhaka,General Physician,Cancelled,4
8,Dhaka,General Physician,No-show,2
9,Dhaka,Gynecologist,Completed,16


In [46]:
# ── Analysis 2: No-show Rate by Division ──────────────────────────────────────
noshow_by_division = (
    df.assign(is_noshow=(df['appointment_status'] == 'No-show'))
      .groupby('division')['is_noshow']
      .mean()
      .mul(100)
      .round(1)
      .reset_index(name='noshow_rate_pct')
      .sort_values('noshow_rate_pct', ascending=False)
)
noshow_by_division.columns = ['division', 'noshow_rate_pct']
print("=== No-show Rate by Division ===")
print(noshow_by_division.to_string(index=False))

=== No-show Rate by Division ===
  division  noshow_rate_pct
     Dhaka             14.6
Chattogram             13.4
  Rajshahi             11.7
    Sylhet              9.3
Mymensingh              8.0
    Khulna              6.7
  Barishal              4.5
   Rangpur              0.0


In [47]:
# ── Analysis 3: Most In-Demand Specialties ────────────────────────────────────
specialty_demand = df['doctor_specialty'].value_counts().reset_index(name='appointments_count')
print("=== Specialty Demand ===")
print(specialty_demand.to_string(index=False))
print()

=== Specialty Demand ===
 doctor_specialty  appointments_count
     Pediatrician                  92
    Dermatologist                  89
     Cardiologist                  85
General Physician                  84
     Gynecologist                  75
       Orthopedic                  75



### Analysis 4: 
Analyses of Wait Days by Specialty of each division

In [48]:
avg_wait = (
    df.groupby(['division', 'doctor_specialty'])['wait_days']
    .agg(
        avg_wait_days='mean',
        median_wait_days='median',
        count='count',
        min_wait_days='min',
        max_wait_days='max',
        standard_deviation='std'
    )
    .round({'avg_wait_days': 1, 'median_wait_days': 1, 'standard_deviation': 1})
    .sort_values(by='avg_wait_days', ascending=False)
    .reset_index()
)
avg_wait.dropna(subset=['standard_deviation'], inplace=True)
display(avg_wait)

,division,doctor_specialty,avg_wait_days,median_wait_days,count,min_wait_days,max_wait_days,standard_deviation
0,Barishal,General Physician,17.7,17.0,3,17,19,1.2
2,Khulna,Gynecologist,15.2,17.0,5,9,20,4.5
4,Rangpur,General Physician,14.2,16.5,4,4,20,7.1
5,Mymensingh,Gynecologist,13.3,15.0,6,1,20,7.4
6,Sylhet,General Physician,13.3,16.0,9,2,20,6.7
7,Rajshahi,Gynecologist,12.9,16.0,10,3,19,6.5
8,Dhaka,Orthopedic,12.3,12.0,22,3,20,4.8
9,Mymensingh,Orthopedic,12.3,12.0,3,11,14,1.5
10,Khulna,Dermatologist,12.0,16.0,11,3,19,6.3
11,Sylhet,Orthopedic,11.8,11.5,12,2,20,5.7


In [49]:
# ── Analysis 5: Age Group Distribution ───────────────────────────────────────
age_dist = df['age_group'].value_counts().sort_index().reset_index(name='count')
print("=== Patient Age Group Distribution ===")
print(age_dist.to_string(index=False))
print()

=== Patient Age Group Distribution ===
          age_group  count
       Child (0-17)     90
Young Adult (18-35)    118
 Middle Age (36-55)    130
       Senior (56+)    162



In [50]:
# ── Analysis 6: Monthly Appointment Trend ─────────────────────────────────────
monthly_trend = (
    df.groupby(['year', 'month', 'month_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['year', 'month'])
)
print("=== Monthly Appointment Trend ===")
print(monthly_trend.to_string(index=False))

=== Monthly Appointment Trend ===
 year  month month_name  count
 2023      1        Jan     23
 2023      2        Feb     15
 2023      3        Mar     25
 2023      4        Apr     16
 2023      5        May     26
 2023      6        Jun     24
 2023      7        Jul     16
 2023      8        Aug     15
 2023      9        Sep     15
 2023     10        Oct     23
 2023     11        Nov     29
 2023     12        Dec     18
 2024      1        Jan     21
 2024      2        Feb     22
 2024      3        Mar     23
 2024      4        Apr     18
 2024      5        May     14
 2024      6        Jun     23
 2024      7        Jul     22
 2024      8        Aug     22
 2024      9        Sep     22
 2024     10        Oct     24
 2024     11        Nov     23
 2024     12        Dec     21


In [51]:
# ── Key Findings Summary ───────────────────────────────────────────────────────
print("\n=== Key Findings ===")
top_noshow = noshow_by_division.iloc[0]
top_specialty = specialty_demand.iloc[0]
longest_wait = avg_wait.iloc[0]
completion_rate = round(status_counts.get('Completed', 0) / len(df) * 100, 1)
top_age_group = age_dist.iloc[3]

print(f"• Overall completion rate        : {completion_rate}%")
print(f"• Division with highest no-show  : {top_noshow['division']} ({top_noshow['noshow_rate_pct']}%)")
print(f"• Most visited specialty         : {top_specialty['doctor_specialty']} ({top_specialty['appointments_count']} appointments)")
print(f"• Longest average wait           : {longest_wait['division']} - {longest_wait['doctor_specialty']} ({longest_wait['avg_wait_days']} days)")
print(f"• Largest patient group          : {top_age_group['age_group']}")


=== Key Findings ===
• Overall completion rate        : 0.0%
• Division with highest no-show  : Dhaka (14.6%)
• Most visited specialty         : Pediatrician (92 appointments)
• Longest average wait           : Barishal - General Physician (17.7 days)
• Largest patient group          : Senior (56+)


In [52]:
# ── Save Results for visualization.ipynb ───────────────────────────────────────
results = {
    'df': df,
    'status_counts': status_counts,
    'noshow_by_division': noshow_by_division,
    'specialty_demand': specialty_demand,
    'avg_wait': avg_wait,
    'age_dist': age_dist,
    'monthly_trend': monthly_trend,
}

with open('analysis_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("Analysis results saved to analysis_results.pkl")

Analysis results saved to analysis_results.pkl
